# 02 — Wants baseline (EDA + TF-IDF baseline)

INTENT-2026-11-11-03: first ML pass over structured wants extracted from each AI agent's sessions.
EDA over intensity / cluster / agent, then a small sklearn baseline that predicts a want's cluster from its title.

Seed of the Muse-side role: *decide what to do from data* (wants -> intent -> order).

Data: `data/bronze/wants.jsonl` (12 rows: Grok 6 + Muse 6), `data/bronze/sessions.jsonl` (index 16).

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path("..")
rows = [json.loads(l) for l in (ROOT/"data/bronze/wants.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
df = pd.DataFrame(rows)
print("shape:", df.shape)
df.head()

## 1. EDA — counts

In [ ]:
print("by cluster:")
print(df.groupby("cluster").size().sort_values(ascending=False))
print()
print("by agent (extracted_by):")
print(df.groupby("extracted_by").size())
print()
print("intensity: mean=%.3f min=%.2f max=%.2f" % (df.intensity.mean(), df.intensity.min(), df.intensity.max()))
print()
print("status:")
print(df.status.value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,4))
df.groupby("cluster").size().sort_values().plot.barh(ax=axes[0], title="wants by cluster")
axes[0].set_xlabel("count")
df.intensity.plot.hist(ax=axes[1], bins=6, title="intensity distribution", range=(0.5,1.0))
axes[1].set_xlabel("intensity")
plt.tight_layout()
plt.show()

## 2. depends_on / enables graph

In [ ]:
edges = []
for r in rows:
    for e in r.get("enables", []):
        edges.append((r["id"], e, "enables"))
    for d in r.get("depends_on", []):
        edges.append((r["id"], d, "depends_on"))
print("edges:", len(edges))
for e in edges:
    print(e)

## 3. Baseline — title -> cluster (TF-IDF + Logistic Regression, leave-one-out)

Only 12 rows, so evaluation is leave-one-out; numbers are not to be over-trusted.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X = df["title"].astype(str).tolist()
y = df["cluster"].astype(str).tolist()
vec = TfidfVectorizer(analyzer="char", ngram_range=(2,4))
Xv = vec.fit_transform(X)
clf = LogisticRegression(max_iter=1000, C=1.0)
pred = cross_val_predict(clf, Xv, y, cv=LeaveOneOut())
acc = accuracy_score(y, pred)
print("LOO accuracy: %.3f (n=%d)" % (acc, len(y)))
print()
labels = sorted(set(y))
print("confusion matrix (rows=actual, cols=pred), labels=", labels)
print(confusion_matrix(y, pred, labels=labels))
print()
print(classification_report(y, pred, zero_division=0))
print("per-sample:")
for i,(t,p,a) in enumerate(zip(X, pred, y)):
    print("  %s pred=%-14s actual=%-14s %s" % (df.id.iloc[i], p, a, "OK" if p==a else "MISS"))

## 4. Result summary

- n = 12 (Grok 6 + Muse 6); clusters: ml-data 5, meta-process 4, agents 2, business 1.
- intensity mean 0.80; max W-20261111-06 (0.95, meta-process).
- depends_on/enables edges: 14.
- Baseline LOO accuracy: 0.25 (3/12). char TF-IDF + LR is not enough at n=12; ml-data dominates predictions.
- Conclusion: baseline is a placeholder; real signal needs more wants (INTENT-02/03 scale-up) and richer features (related/depends_on graph, agent). Numbers reported without over-confidence.